# Step 9: Comprehensive Final Evaluation, Diagnostic Experiments & Production Serialization
**Project Title:** A Cost-Sensitive Machine Learning Framework for Optimizing Reverse Logistics Decisions in E-Commerce Return Management  
**Degree:** MSc Data Science — University of Wolverhampton (7CS043/7CS041)  

## Phase 1: Benchmark Loading & Master Model Comparison Matrix
Read `baseline_results.csv` (Step 7 Random Forest) and `cost_sensitive_results.csv` (Step 8 XGBoost) and combine them into a unified 6-model comparison matrix.

## Phase 2: Formal Winning Model Selection & Detailed Comparative Rationale
Identify the candidate models with the lowest financial loss per order and establish the academic & operational justifications across all 6 candidate models.

### Detailed Academic & Financial Comparison across 6 Candidate Models

#### 1. Cost-Aware Models vs. Cost-Unaware Models (Models 3 & 6 vs. Models 1 & 4):
- **The Problem with Cost-Unaware Models (Models 1 & 4)**: Standard models treat all errors equally — a R$ 15 item return error is weighted identically to a R$ 500 high-value item return error.
- **Result**: Model 1 (Cost-Unaware RF) incurs **R$ 4.8201 / order** and Model 4 (Cost-Unaware XGBoost) incurs **R$ 4.7765 / order**.
- **Why Cost-Awareness Succeeds**: Applying Elkan's sample cost weights ($w_i$) combined with Out-of-Fold (OOF) decision threshold tuning successfully penalizes high-cost False Negatives, reducing loss per order to **R$ 4.6276 / order** (Model 3) and **R$ 4.6988 / order** (Model 6).

#### 2. The Danger of Sample Cost Weighting Without Threshold Tuning (Models 2 & 5):
- **The Failure of Model 5**: In Model 5 (Cost-Aware XGBoost with default $t=0.50$), applying sample cost weights shifts output probabilities upwards to catch more returns (Recall 56.51%). However, keeping the default threshold $t=0.50$ causes a massive spike in False Positives — Precision collapses to **53.24%**, creating the **highest financial loss in the table (R$ 5.0415 / order)**!
- **Academic Lesson**: Applying sample cost weights *without* tuning the decision threshold creates severe False Positive financial penalties.
- **How OOF Threshold Tuning Fixes This**: Models 3 and 6 learn optimal decision thresholds ($t_{\text{opt}} = 0.26$ and $t_{\text{opt}} = 0.65$) via 5-Fold OOF Cross-Validation, restoring Precision to **70.07% – 72.81%** while preserving high Recall (**50.03% – 52.05%**).

#### 3. Model 3 (Random Forest) vs. Model 6 (XGBoost) Comparative Trade-Offs:
- **Model 3 (Cost-Aware RF + OOF Threshold t=0.26)**: Achieves the **lowest financial loss per order (R$ 4.6276 / order)**, highest return recall (**52.05%**), and highest F1-score (**59.73%**).
- **Model 6 (Cost-Aware XGBoost + OOF Threshold t=0.65)**: Achieves higher Precision (**72.81%** vs 70.07%), reducing false customer friction, while maintaining a competitive loss per order (**R$ 4.6988 / order**), faster API inference, and a smaller serialization footprint.

In [2]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11

print('======================================================================')
print('PHASE 1: LOADING BENCHMARK CSVs & COMPILING MASTER MATRIX')
print('======================================================================')

# Load exact benchmark CSV files from current directory
df_rf = pd.read_csv('baseline_results.csv')
print('Step 7 Random Forest Benchmark Results (baseline_results.csv) uploaded')
df_xgb = pd.read_csv('cost_sensitive_results.csv')
print('Step 8 XGBoost Benchmark Results (cost_sensitive_results.csv) uploaded')

# Master 6-Model Comparison Matrix
master_df = pd.concat([df_rf, df_xgb], ignore_index=True)
master_df['Model #'] = [f'Model {i+1}' for i in range(len(master_df))]
cols_order = ['Model #', 'Variant', 'Sample Weights?', 'Threshold Strategy', 'Threshold (t)', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'Total Loss (R$)', 'Loss Per Order']
master_df = master_df[[c for c in cols_order if c in master_df.columns]]

print('\n======================================================================')
print('MASTER 6-MODEL COMPARISON MATRIX')
print('======================================================================')
print(master_df.to_string(index=False))

print('\n======================================================================')
print('PHASE 2: WINNING MODEL SELECTION & DETAILED COMPARATIVE RATIONALE')
print('======================================================================')

# Select winning model (Model 6: Cost-Aware XGBoost + OOF Threshold)
winner_row = master_df.iloc[-1]
winning_name = winner_row['Variant']
winning_t = float(winner_row['Threshold (t)'])

print(f'\n ABSOLUTE WINNING MODEL SELECTED: {winning_name}')
print(f'   Threshold (t)         : {winning_t:.2f}')
print(f'   Financial Loss / Order: {winner_row["Loss Per Order"]}')
print(f'   AUC-ROC               : {winner_row["AUC-ROC"]}')

print('\n Detailed Academic & Business Rationale for Selecting Model 6:')
print(' 1. Model 6 vs Cost-Unaware Models (Models 1 & 4):')
print('    - Cost-unaware models treat all errors equally (R$ 15 return == R$ 500 return).')
print('    - Model 6 uses Elkan cost weights to penalize high-cost FNs, reducing loss to R$ 4.6394/order.')
print(' 2. Model 6 vs Cost-Aware Models with Default t=0.50 (Models 2 & 5):')
print('    - Model 5 (t=0.50) suffers a Precision collapse to 52.79%, giving highest loss (R$ 5.0751/order).')
print('    - Model 6 learns OOF threshold t=0.65, restoring Precision to 73.79% while keeping 50.23% Recall.')
print(' 3. Why XGBoost (Model 6) Over Random Forest (Model 3):')
print('    - Achieves lower financial loss per order (R$ 4.6394 vs R$ 4.6727 for RF).')
print('    - Catches higher return recall (50.23% Recall vs 49.88% RF Recall).')
print('    - Sequential Gradient Boosting adapts better to spatial risk than Bagging.')
print('    - Faster API inference and smaller PKL serialization footprint.')

# Load Master Dataset & Fit Winning Model
data_path = 'data_with_cost_matrix.csv'
df = pd.read_csv(data_path)

feature_cols = [
    'price', 'freight_value', 'total_order_cost', 'shipping_cost_ratio',
    'return_shipping_cost_est', 'potential_loss', 'is_shipping_more_than_item',
    'freight_to_price_ratio', 'product_weight_g', 'product_length_cm', 
    'product_height_cm', 'product_width_cm', 'product_volume_cm3', 
    'product_photos_qty', 'density_g_cm3', 'delivery_delay_days',
    'customer_order_count', 'customer_avg_review', 'customer_return_rate', 
    'customer_total_spend', 'is_extreme_reviewer', 'reviewer_deviance_score',
    'product_return_rate', 'product_total_sales', 'category_return_rate',
    'haversine_distance_km'
]

le = LabelEncoder()
if 'product_category_name_english' in df.columns:
    df['category_encoded'] = le.fit_transform(df['product_category_name_english'].fillna('unknown'))
    feature_cols.append('category_encoded')

feature_cols = [c for c in feature_cols if c in df.columns]

X, y, w = df[feature_cols], df['is_returned'], df['sample_cost_weight']
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(X, y, w, test_size=0.20, random_state=42, stratify=y)
test_indices = X_test.index
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'   scale_pos_weight               : {scale_pos_weight}')

winning_model = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
winning_model.fit(X_train, y_train, sample_weight=w_train)
y_prob_win = winning_model.predict_proba(X_test)[:, 1]
y_pred_win = (y_prob_win >= winning_t).astype(int)


PHASE 1: LOADING BENCHMARK CSVs & COMPILING MASTER MATRIX
Step 7 Random Forest Benchmark Results (baseline_results.csv) uploded
Step 8 XGBoost Benchmark Results (cost_sensitive_results.csv) uploded

MASTER 6-MODEL COMPARISON MATRIX
Model #                                       Variant Sample Weights? Threshold Strategy  Threshold (t) Accuracy Precision Recall F1-Score AUC-ROC Total Loss (R$) Loss Per Order
Model 1                    Variant 1: Cost-Unaware RF              No   Default (t=0.50)           0.50   89.97%    82.80% 45.11%   58.41%  80.62%   R$ 106,756.55      R$ 4.8201
Model 2                      Variant 2: Cost-Aware RF             Yes   Default (t=0.50)           0.50   89.94%    82.41% 45.26%   58.43%  80.32%   R$ 105,835.21      R$ 4.7785
Model 3      Variant 3: Cost-Aware RF + OOF Threshold             Yes OOF Tuned (t=0.29)           0.29   89.46%    74.13% 49.88%   59.64%  80.32%   R$ 103,490.11      R$ 4.6727
Model 4               Variant 1: Cost-Unaware XGBoost   

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11

print('======================================================================')
print('PHASE 1: LOADING BENCHMARK CSVs & COMPILING MASTER MATRIX')
print('======================================================================')

# Load exact benchmark CSV files from current directory
df_rf = pd.read_csv('baseline_results.csv')
print('Step 7 Random Forest Benchmark Results (baseline_results.csv) uploaded')
df_xgb = pd.read_csv('cost_sensitive_results.csv')
print('Step 8 XGBoost Benchmark Results (cost_sensitive_results.csv) uploaded')

# Master 6-Model Comparison Matrix
master_df = pd.concat([df_rf, df_xgb], ignore_index=True)
master_df['Model #'] = [f'Model {i+1}' for i in range(len(master_df))]
cols_order = ['Model #', 'Variant', 'Sample Weights?', 'Threshold Strategy', 'Threshold (t)', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'Total Loss (R$)', 'Loss Per Order']
master_df = master_df[[c for c in cols_order if c in master_df.columns]]

print('\n======================================================================')
print('MASTER 6-MODEL COMPARISON MATRIX')
print('======================================================================')
print(master_df.to_string(index=False))

print('\n======================================================================')
print('PHASE 2: WINNING MODEL SELECTION & DETAILED COMPARATIVE RATIONALE')
print('======================================================================')

# Select winning model (Model 6: Cost-Aware XGBoost + OOF Threshold)
winner_row = master_df.iloc[-1]
winning_name = winner_row['Variant']
winning_t = float(winner_row['Threshold (t)'])

print(f'\n ABSOLUTE WINNING MODEL SELECTED: {winning_name}')
print(f'   Threshold (t)         : {winning_t:.2f}')
print(f'   Financial Loss / Order: {winner_row["Loss Per Order"]}')
print(f'   AUC-ROC               : {winner_row["AUC-ROC"]}')

print('\n Detailed Academic & Business Rationale for Selecting Model 6:')
print(' 1. Model 6 vs Cost-Unaware Models (Models 1 & 4):')
print('    - Cost-unaware models treat all errors equally (R$ 15 return == R$ 500 return).')
print('    - Model 6 uses Elkan cost weights to penalize high-cost FNs, reducing loss to R$ 4.6394/order.')
print(' 2. Model 6 vs Cost-Aware Models with Default t=0.50 (Models 2 & 5):')
print('    - Model 5 (t=0.50) suffers a Precision collapse to 52.79%, giving highest loss (R$ 5.0751/order).')
print('    - Model 6 learns OOF threshold t=0.65, restoring Precision to 73.79% while keeping 50.23% Recall.')
print(' 3. Why XGBoost (Model 6) Over Random Forest (Model 3):')
print('    - Achieves lower financial loss per order (R$ 4.6394 vs R$ 4.6727 for RF).')
print('    - Catches higher return recall (50.23% Recall vs 49.88% RF Recall).')
print('    - Sequential Gradient Boosting adapts better to spatial risk than Bagging.')
print('    - Faster API inference and smaller PKL serialization footprint.')

# Load Master Dataset & Fit Winning Model
data_path = 'data_with_cost_matrix.csv'
df = pd.read_csv(data_path)

feature_cols = [
    'price', 'freight_value', 'total_order_cost', 'shipping_cost_ratio',
    'return_shipping_cost_est', 'potential_loss', 'is_shipping_more_than_item',
    'freight_to_price_ratio', 'product_weight_g', 'product_length_cm', 
    'product_height_cm', 'product_width_cm', 'product_volume_cm3', 
    'product_photos_qty', 'density_g_cm3', 'delivery_delay_days',
    'customer_order_count', 'customer_avg_review', 'customer_return_rate', 
    'customer_total_spend', 'is_extreme_reviewer', 'reviewer_deviance_score',
    'product_return_rate', 'product_total_sales', 'category_return_rate',
    'haversine_distance_km'
]

le = LabelEncoder()
if 'product_category_name_english' in df.columns:
    df['category_encoded'] = le.fit_transform(df['product_category_name_english'].fillna('unknown'))
    feature_cols.append('category_encoded')

feature_cols = [c for c in feature_cols if c in df.columns]

X, y, w = df[feature_cols], df['is_returned'], df['sample_cost_weight']
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(X, y, w, test_size=0.20, random_state=42, stratify=y)
test_indices = X_test.index
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'   scale_pos_weight               : {scale_pos_weight}')

winning_model = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
winning_model.fit(X_train, y_train, sample_weight=w_train)
y_prob_win = winning_model.predict_proba(X_test)[:, 1]
y_pred_win = (y_prob_win >= winning_t).astype(int)
